# AndinaLog 03B — Notebook 2: tratamiento IoT didáctico v2

Lee el diagnosticado completo, conserva el original y añade valores tratados,
banderas y motivos. Silver y cuarentena final reparten todas las filas. La
hora de origen y de análisis es `America/La_Paz`; no se deriva UTC.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ENTORNO = "auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
MAX_INTERVALO_VECINOS_MIN = 60
MAX_CAMBIO_TEMPERATURA_C = 2.0
MAX_CAMBIO_HUMEDAD_PCT = 20.0
COLUMNAS_BRONZE = ["timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
    "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct",
    "desviacion_termica_flag", "desviacion_proximos_60min_flag"]

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
        if not (raiz / "proyecto-integrador/01_diagnostico/andinalog_iot_telemetry/salidas/andinalog_iot_telemetry_didactico_v2_diagnosticado.csv").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "proyecto-integrador/01_diagnostico/andinalog_iot_telemetry/salidas/andinalog_iot_telemetry_didactico_v2_diagnosticado.csv").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ = encontrar_raiz()
ENTRADA = RAIZ / "proyecto-integrador/01_diagnostico/andinalog_iot_telemetry/salidas/andinalog_iot_telemetry_didactico_v2_diagnosticado.csv"
SALIDAS = RAIZ / "proyecto-integrador/02_tratamiento/andinalog_iot_telemetry/salidas"
df = pd.read_csv(ENTRADA, dtype="string", encoding="utf-8-sig", keep_default_na=False)
requeridas = ["fila_bronze", "en_cuarentena", "zona_horaria_origen", *COLUMNAS_BRONZE]
requeridas += [f"{c}_{s}" for c in COLUMNAS_BRONZE for s in ("en_cuarentena", "motivo")]
faltantes = sorted(set(requeridas) - set(df.columns))
if faltantes: raise ValueError(f"Faltan columnas del diagnóstico: {faltantes}")
if df["fila_bronze"].duplicated().any(): raise ValueError("fila_bronze debe ser único")
if not df["zona_horaria_origen"].eq("America/La_Paz").all():
    raise ValueError("La zona de origen debe ser America/La_Paz")
if not df["en_cuarentena"].isin(["True", "False"]).all():
    raise ValueError("en_cuarentena debe ser True o False")
if not df["en_cuarentena"].eq(df[[f"{c}_en_cuarentena" for c in COLUMNAS_BRONZE]].eq("True").any(axis=1).astype(str)).all():
    raise ValueError("La cuarentena por fila no coincide con las banderas por columna")
original = df[requeridas].copy(deep=True)
print("Filas diagnosticadas:", len(df))


## 1. Correcciones deterministas

Las columnas `*_tratado` son nuevas. Las originales y las marcas del diagnóstico no se sobrescriben. Los motivos y acciones del tratamiento se agregan a nivel de fila.


In [ ]:
df["acciones_tratamiento"] = ""
df["motivos_tratamiento"] = ""
def anotar(mascara, accion, motivo=""):
    mascara = pd.Series(mascara, index=df.index).fillna(False).astype(bool)
    for campo, texto in [("acciones_tratamiento", accion), ("motivos_tratamiento", motivo)]:
        if texto:
            previo = df.loc[mascara, campo]
            df.loc[mascara, campo] = previo.where(previo.eq(""), previo + " | ") + texto

df["camion_id_tratado"] = df["camion_id"].str.strip().str.upper()
camion_normalizado = df["camion_id_tratado"].ne(df["camion_id"]) & df["camion_id_tratado"].str.fullmatch(r"CAM-\d{2}").fillna(False)
anotar(camion_normalizado, "NORMALIZAR_CAMION_ID", "Mayúsculas y espacios normalizados")

df["temp_unit_tratado"] = df["temp_unit"].str.strip().str.upper()
temp_raw = pd.to_numeric(df["temperatura_cabina_c"], errors="coerce")
centinela = temp_raw.eq(-999)
temp_raw = temp_raw.mask(centinela)
df["temperatura_cabina_c_tratada"] = temp_raw.where(df["temp_unit_tratado"].eq("C"),
    (temp_raw - 32) * 5 / 9).where(df["temp_unit_tratado"].isin(["C", "F"]))
convertida = df["temp_unit_tratado"].eq("F") & df["temperatura_cabina_c_tratada"].notna()
df["temperatura_convertida_f_a_c"] = convertida
anotar(convertida, "CONVERTIR_F_A_C", "Fahrenheit convertido a Celsius")
anotar(centinela, "CENTINELA_A_FALTANTE", "-999 no representa temperatura")
df["temp_unit_tratado"] = df["temp_unit_tratado"].mask(df["temp_unit_tratado"].isin(["C", "F"]), "C")

humedad_raw = pd.to_numeric(df["humedad_cabina_pct"], errors="coerce")
df["humedad_cabina_pct_tratada"] = humedad_raw.where(humedad_raw.between(0, 100))

fecha_local = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
df["timestamp_bolivia_tratado"] = fecha_local.dt.strftime("%Y-%m-%d %H:%M:%S").fillna("")


## 2. Imputación restringida

Se interpolan faltantes interiores solamente. Los dos vecinos deben ser **observaciones válidas**, del mismo `viaje_id`, `camion_id_tratado` y `producto_id`, separados por hasta 60 minutos, con banderas térmicas `0` y sin duplicidad de clave. Se exige además una variación máxima de 2 °C o 20 puntos de humedad entre vecinos. Esos límites son parámetros conservadores del ejercicio y deben revisarse con el responsable del sensor. No se interpola sobre una excursión ni se usa una fila ya imputada como vecina.


In [ ]:
# Duplicados exactos y conflictos de clave se deciden con el Bronze original.
firma = pd.util.hash_pandas_object(df[COLUMNAS_BRONZE], index=False)
clave = [df["viaje_id"], df["timestamp"]]
variantes = firma.groupby(clave, dropna=False).transform("nunique")
clave_repetida = df.duplicated(["viaje_id", "timestamp"], keep=False)
copia = clave_repetida & variantes.eq(1) & df.duplicated(COLUMNAS_BRONZE, keep="first")
conflicto = clave_repetida & variantes.gt(1)
anotar(copia, "EXCLUIR_COPIA", "Copia exacta de otra lectura")
anotar(conflicto, "CUARENTENA_CONFLICTO", "Misma clave de lectura con datos distintos")

df["temperatura_imputada"] = False
df["humedad_imputada"] = False
grupos = ["viaje_id", "camion_id_tratado", "producto_id"]
orden = df.assign(_fecha=fecha_local).sort_values(grupos + ["_fecha", "fila_bronze"], kind="stable")
orden = orden.loc[orden["_fecha"].notna()].copy()
g = orden.groupby(grupos, sort=False, dropna=False)
anterior = g["_fecha"].shift(1)
siguiente = g["_fecha"].shift(-1)
duracion = (siguiente - anterior).dt.total_seconds() / 60
interior = anterior.lt(orden["_fecha"]) & orden["_fecha"].lt(siguiente) & duracion.le(MAX_INTERVALO_VECINOS_MIN)
sin_conflicto = ~(copia | conflicto)
vecinos_sin_conflicto = sin_conflicto.reindex(orden.index).groupby([orden[c] for c in grupos], dropna=False).shift(1).fillna(False).astype(bool) & sin_conflicto.reindex(orden.index).groupby([orden[c] for c in grupos], dropna=False).shift(-1).fillna(False).astype(bool)
flags_cero = orden["desviacion_termica_flag"].eq("0") & orden["desviacion_proximos_60min_flag"].eq("0")
vecinos_flags_cero = flags_cero.groupby([orden[c] for c in grupos], dropna=False).shift(1).fillna(False).astype(bool) & flags_cero.groupby([orden[c] for c in grupos], dropna=False).shift(-1).fillna(False).astype(bool)
base = interior & sin_conflicto.reindex(orden.index) & vecinos_sin_conflicto & flags_cero & vecinos_flags_cero
fraccion = (orden["_fecha"] - anterior).dt.total_seconds() / (siguiente - anterior).dt.total_seconds()

def interpolar(columna, max_cambio, marca, accion):
    valor = pd.to_numeric(orden[columna], errors="coerce")
    antes = valor.groupby([orden[c] for c in grupos], dropna=False).shift(1)
    despues = valor.groupby([orden[c] for c in grupos], dropna=False).shift(-1)
    elegible = base & valor.isna() & antes.notna() & despues.notna() & (despues - antes).abs().le(max_cambio)
    if columna.startswith("temperatura"):
        unidad_observada = orden["temp_unit"].str.strip().str.upper()
        unidades_vecinas = unidad_observada.groupby([orden[c] for c in grupos], dropna=False)
        elegible &= unidad_observada.eq("C") & unidades_vecinas.shift(1).isin(["C", "F"]) & unidades_vecinas.shift(-1).isin(["C", "F"])
    nuevos = antes + fraccion * (despues - antes)
    indices = orden.index[elegible.fillna(False)]
    df.loc[indices, columna] = nuevos.loc[indices].round(3)
    df.loc[indices, marca] = True
    anotar(df.index.isin(indices), accion, "Interpolación entre dos lecturas observadas cercanas y coherentes")

interpolar("temperatura_cabina_c_tratada", MAX_CAMBIO_TEMPERATURA_C, "temperatura_imputada", "IMPUTAR_TEMPERATURA")
interpolar("humedad_cabina_pct_tratada", MAX_CAMBIO_HUMEDAD_PCT, "humedad_imputada", "IMPUTAR_HUMEDAD")


## 3. Decisión final y uso analítico

Una fila recuperada puede pasar a datos tratados aunque el diagnóstico original dijera cuarentena; ambas decisiones quedan disponibles. Kelvin, fecha inválida, identificador indispensable inválido, conflicto de clave o magnitud sin recuperación van a cuarentena final. Las claves ausentes en los maestros se conservan como `NO_EVALUABLE`, sin convertir por sí solas la lectura en cuarentena.

`apta_kpi_termico` usa Productos Silver didáctico actual y excluye temperaturas imputadas, umbrales inferidos o no verificables y banderas inconsistentes. La posible incompatibilidad `Fresco`/`Congelado` en camión `Seco` es una alerta de revisión, no un motivo automático de cuarentena ni de exclusión del KPI térmico. `apta_objetivo_60min` excluye filas con temperatura imputada para evitar utilizarla como observación de entrenamiento.


In [ ]:
formato_ids = (
    df["viaje_id"].str.fullmatch(r"VIA-\d{5}").fillna(False) &
    df["order_id"].str.fullmatch(r"ORD-\d{4}-\d{5}").fillna(False) &
    df["camion_id_tratado"].str.fullmatch(r"CAM-\d{2}").fillna(False) &
    df["producto_id"].str.fullmatch(r"PROD-\d{3}").fillna(False))
flags_validos = df["desviacion_termica_flag"].isin(["0", "1"]) & df["desviacion_proximos_60min_flag"].isin(["0", "1"])
unidad_valida = df["temp_unit"].str.strip().str.upper().isin(["C", "F"])
df["motivo_cuarentena_final"] = ""
def cuarentena_si(mascara, motivo):
    mascara = pd.Series(mascara, index=df.index).fillna(True).astype(bool)
    previo = df.loc[mascara, "motivo_cuarentena_final"]
    df.loc[mascara, "motivo_cuarentena_final"] = previo.where(previo.eq(""), previo + " | ") + motivo

cuarentena_si(fecha_local.isna(), "Fecha local inválida")
cuarentena_si(~formato_ids, "Identificador indispensable inválido")
cuarentena_si(~flags_validos, "Bandera inválida")
cuarentena_si(~unidad_valida, "Unidad térmica no operacional o desconocida")
cuarentena_si(df["temperatura_cabina_c_tratada"].isna(), "Temperatura sin valor recuperable")
cuarentena_si(df["humedad_cabina_pct_tratada"].isna(), "Humedad sin valor recuperable")
cuarentena_si(copia, "Copia exacta excluida")
cuarentena_si(conflicto, "Clave de lectura en conflicto")
df["en_cuarentena_final"] = df["motivo_cuarentena_final"].ne("")
df["decision_tratamiento"] = np.where(df["en_cuarentena_final"], "CUARENTENA", "SILVER")

# Evaluación con los maestros didácticos actuales. Las claves ausentes se señalan,
# pero no cambian la decisión de cuarentena técnica de una lectura recuperable.
ruta_productos = RAIZ / "proyecto-integrador/02_tratamiento/andinalog_productos/salidas/andinalog_productos_didactico_v1_silver.csv"
ruta_flota = RAIZ / "proyecto-integrador/02_tratamiento/andinalog_flota/salidas/andinalog_flota_didactico_v1_silver.csv"
for ruta in (ruta_productos, ruta_flota):
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta maestro didáctico para evaluar la lectura: {ruta}")
productos = pd.read_csv(ruta_productos, dtype="string", encoding="utf-8-sig", keep_default_na=False)
flota = pd.read_csv(ruta_flota, dtype="string", encoding="utf-8-sig", keep_default_na=False)
if productos["producto_id_tratado"].duplicated().any():
    raise ValueError("producto_id_tratado duplicado en Productos Silver")
if flota["camion_id_tratado"].duplicated().any():
    raise ValueError("camion_id_tratado duplicado en Flota Silver")
p = productos.set_index("producto_id_tratado")
f = flota.set_index("camion_id_tratado")
objetivo = pd.to_numeric(df["producto_id"].map(p["temperatura_conservacion_requerida_c_tratada"]), errors="coerce")
tolerancia = pd.to_numeric(df["producto_id"].map(p["tolerancia_temperatura_c_tratada"]), errors="coerce")
umbral_observado = df["producto_id"].map(p["apto_umbral_termico_observado"]).eq("True").fillna(False)
df["umbral_producto_evaluable"] = (objetivo.notna() & tolerancia.ge(0) & umbral_observado).fillna(False)
fuera = (pd.to_numeric(df["temperatura_cabina_c_tratada"], errors="coerce") - objetivo).abs().gt(tolerancia)
df["flag_termico_coherente"] = (df["umbral_producto_evaluable"] &
    fuera.eq(df["desviacion_termica_flag"].eq("1"))).fillna(False)
df["apta_kpi_termico"] = (~df["en_cuarentena_final"] & ~df["temperatura_imputada"] &
    df["umbral_producto_evaluable"] & df["flag_termico_coherente"]).fillna(False)

# Alerta relacional de negocio. Se evalúa solo con categoría observada y
# tipo de camión documentado; no corrige datos ni envía filas a cuarentena.
categoria = df["producto_id"].map(p["categoria_logistica_tratada"])
categoria_observada = df["producto_id"].map(p["categoria_imputada"]).eq("False").fillna(False)
tipo_camion = df["camion_id_tratado"].map(f["tipo_camion_tratado"])
df["categoria_producto_maestro"] = categoria.fillna("").astype("string")
df["tipo_camion_maestro"] = tipo_camion.fillna("").astype("string")
df["compatibilidad_tipo_camion_estado"] = "NO_EVALUABLE"
evaluable = categoria_observada & categoria.isin(["Fresco", "Congelado", "Seco"]).fillna(False) & tipo_camion.isin(["Seco", "Refrigerado"]).fillna(False)
df.loc[evaluable, "compatibilidad_tipo_camion_estado"] = "SIN_ALERTA"
df["posible_incompatibilidad_termica"] = (evaluable & categoria.isin(["Fresco", "Congelado"]).fillna(False) & tipo_camion.eq("Seco").fillna(False))
df.loc[df["posible_incompatibilidad_termica"], "compatibilidad_tipo_camion_estado"] = "REVISAR"
df["compatibilidad_tipo_camion_motivo"] = ""
df.loc[df["posible_incompatibilidad_termica"], "compatibilidad_tipo_camion_motivo"] = "Producto Fresco/Congelado en camion registrado Seco; verificar configuracion operativa"
df.loc[~evaluable, "compatibilidad_tipo_camion_motivo"] = "Categoria observada o tipo de camion no disponible para evaluar"
df["apta_objetivo_60min"] = ~df["en_cuarentena_final"] & ~df["temperatura_imputada"] & flags_validos
tratado = df.loc[~df["en_cuarentena_final"]].copy()
cuarentena_final = df.loc[df["en_cuarentena_final"]].copy()


## 4. Comprobaciones y exportación

Se exportan tres CSV: Silver, cuarentena final y reporte de calidad. Las
columnas Bronze y las banderas diagnósticas se conservan. `en_cuarentena`
describe la etapa diagnóstica; `en_cuarentena_final` describe el destino
posterior al tratamiento.


In [ ]:
pd.testing.assert_frame_equal(df[requeridas], original)
assert df["fila_bronze"].is_unique
assert len(df) == len(tratado) + len(cuarentena_final)
assert cuarentena_final["motivo_cuarentena_final"].ne("").all()
assert tratado["motivo_cuarentena_final"].eq("").all()
assert not (tratado["apta_kpi_termico"] & tratado["temperatura_imputada"]).any()
assert not (tratado["apta_kpi_termico"] & ~tratado["umbral_producto_evaluable"]).any()
assert tratado["posible_incompatibilidad_termica"].eq(tratado["compatibilidad_tipo_camion_estado"].eq("REVISAR")).all()
assert not tratado.loc[tratado["posible_incompatibilidad_termica"], "en_cuarentena_final"].any()
assert not (tratado["apta_objetivo_60min"] & tratado["temperatura_imputada"]).any()
assert not tratado["timestamp_bolivia_tratado"].eq("").any()
assert tratado["temp_unit_tratado"].eq("C").all()
assert not tratado["temperatura_cabina_c_tratada"].isna().any()
assert not tratado["humedad_cabina_pct_tratada"].isna().any()
assert not tratado["camion_id_tratado"].str.fullmatch(r"CAM-\d{2}").fillna(False).eq(False).any()

SALIDAS.mkdir(parents=True, exist_ok=True)
ruta_tratado = SALIDAS / "andinalog_iot_telemetry_didactico_v2_silver.csv"
ruta_cuarentena = SALIDAS / "andinalog_iot_telemetry_didactico_v2_cuarentena_final.csv"
ruta_reporte = SALIDAS / "andinalog_iot_telemetry_didactico_v2_reporte_calidad_tratamiento.csv"
silver_export = tratado.rename(columns={"en_cuarentena": "en_cuarentena_diagnostico"})
cuarentena_export = cuarentena_final.rename(columns={"en_cuarentena": "en_cuarentena_diagnostico"})
silver_export.to_csv(ruta_tratado, index=False, encoding="utf-8-sig")
cuarentena_export.to_csv(ruta_cuarentena, index=False, encoding="utf-8-sig")
reporte = pd.DataFrame([
    ("filas_diagnosticadas", len(df)),
    ("filas_cuarentena_diagnostico", int(df["en_cuarentena"].eq("True").sum())),
    ("filas_recuperadas_desde_cuarentena", int(tratado["en_cuarentena"].eq("True").sum())),
    ("filas_silver", len(tratado)),
    ("filas_cuarentena_final", len(cuarentena_final)),
    ("temperaturas_convertidas_f_a_c", int(tratado["temperatura_convertida_f_a_c"].sum())),
    ("temperaturas_imputadas", int(tratado["temperatura_imputada"].sum())),
    ("humedades_imputadas", int(tratado["humedad_imputada"].sum())),
    ("lecturas_aptas_kpi_termico", int(tratado["apta_kpi_termico"].sum())),
    ("lecturas_posible_incompatibilidad", int(tratado["posible_incompatibilidad_termica"].sum())),
], columns=["metrica", "valor"])
reporte.to_csv(ruta_reporte, index=False, encoding="utf-8-sig")
assert len(df) == len(tratado) + len(cuarentena_final)
assert "timestamp_utc" not in tratado.columns
display(reporte)
print("Diagnosticadas:", len(df), "| Tratadas:", len(tratado), "| Cuarentena final:", len(cuarentena_final))
print("F a C:", int(convertida.sum()), "| Camiones normalizados:", int(camion_normalizado.sum()),
      "| Temperaturas imputadas:", int(df["temperatura_imputada"].sum()),
      "| Humedades imputadas:", int(df["humedad_imputada"].sum()))
print("Aptas KPI térmico:", int(tratado["apta_kpi_termico"].sum()))
print("Viajes con posible incompatibilidad:", tratado.loc[tratado["posible_incompatibilidad_termica"], "viaje_id"].nunique())
print("Tratado:", ruta_tratado)
print("Cuarentena:", ruta_cuarentena)
print("Reporte:", ruta_reporte)
display(df[["fila_bronze", "en_cuarentena", "decision_tratamiento", "acciones_tratamiento",
    "motivo_cuarentena_final", "apta_kpi_termico"]].head(10))
